# 02. Baseline model: architecture, training, initial metrics
MobileNetV3-Large fine-tuned on the training split from notebook 01; metrics on the validation split. The test split is not read in this notebook.

In [ ]:
!pip install --quiet torchinfo scikit-learn
import torch, torchvision, pandas as pd, numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
image_directory = "/content/data/train_images"
labels = pd.read_csv("split_v1_seed42.csv")   # upload the file from notebook 01 if this is a new session
EPOCHS, SEED = 3, 42
torch.manual_seed(SEED); np.random.seed(SEED)

In [ ]:
class CassavaDataset(Dataset):
    def __init__(self, frame, transform): self.frame, self.transform = frame.reset_index(drop=True), transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return self.transform(Image.open(f"{image_directory}/{row.image_id}").convert("RGB")), int(row.label)
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_transform = transforms.Compose([transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2), transforms.ToTensor(), normalize])
eval_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), normalize])
train_loader = DataLoader(CassavaDataset(labels[labels.split == "train"], train_transform), batch_size=64, shuffle=True, num_workers=2)
validation_loader = DataLoader(CassavaDataset(labels[labels.split == "validation"], eval_transform), batch_size=128, num_workers=2)

## Model architecture
MobileNetV3-Large: inverted residual blocks with squeeze-and-excitation, hard-swish activations, 1280-feature head replaced by a 5-way linear layer. Loss: class-weighted cross-entropy (CMD is about 60% of labels). Optimiser: AdamW, learning rate 3e-4, cosine decay.

In [ ]:
from torchinfo import summary
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V1")
model.classifier[-1] = torch.nn.Linear(model.classifier[-1].in_features, 5)
model = model.to(device)
print(summary(model, input_size=(1, 3, 224, 224), depth=2, col_names=["output_size", "num_params"]))

In [ ]:
class_counts = torch.tensor(labels[labels.split == "train"]["label"].value_counts().sort_index().values, dtype=torch.float)
criterion = torch.nn.CrossEntropyLoss(weight=(class_counts.sum() / class_counts).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
for epoch in range(EPOCHS):
    model.train(); running = 0.0
    for images, targets in tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}"):
        optimizer.zero_grad(); loss = criterion(model(images.to(device)), targets.to(device)); loss.backward(); optimizer.step(); scheduler.step(); running += loss.item()
    print(f"epoch {epoch + 1}: mean training loss {running / len(train_loader):.4f}")
torch.save(model.state_dict(), "mobilenetv3_baseline.pt"); print("saved mobilenetv3_baseline.pt (copy to models/ in the repository)")

## Initial performance metrics (validation split)
Accuracy, precision, recall and F1 per class, macro-F1, and the confusion matrix. These are uncalibrated; calibration and abstention come next.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
import matplotlib.pyplot as plt
model.eval(); predictions, truth = [], []
with torch.no_grad():
    for images, targets in validation_loader:
        predictions += model(images.to(device)).argmax(1).cpu().tolist(); truth += targets.tolist()
names = ["CBB", "CBSD", "CGM", "CMD", "healthy"]
print("accuracy:", round(accuracy_score(truth, predictions), 4)); print("macro-F1:", round(f1_score(truth, predictions, average="macro"), 4))
print(classification_report(truth, predictions, target_names=names, digits=3))
ConfusionMatrixDisplay(confusion_matrix(truth, predictions), display_labels=names).plot(cmap="Blues", xticks_rotation=45); plt.title("Validation confusion matrix"); plt.tight_layout(); plt.show()